In [1]:
!pip install -q transformers==4.46.0 peft bitsandbytes accelerate jamotools jamo trl -q

In [2]:
import torch
import torchaudio
import io
import os
import jamotools
from jamo import h2j, j2hcj
from transformers import (
    Wav2Vec2ForCTC, Wav2Vec2Processor,
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, AutoModelForSequenceClassification
)
from peft import PeftModel
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU
print(f"✅ GPU: {torch.cuda.is_available()}")
print(f"🖥️ Device: {torch.cuda.get_device_name(0)}")

# Paths
ASR_MODEL_PATH         = "/content/drive/MyDrive/manual_datasets/clovacall_data/final_asr_jamo_model"
RESTAURANT_EXPERT_PATH = "/content/drive/MyDrive/manual_datasets/dialogue_system/restaurant_expert"
TRAVEL_EXPERT_PATH     = "/content/drive/MyDrive/manual_datasets/dialogue_system/travel_expert"
ROUTER_PATH            = "/content/drive/MyDrive/manual_datasets/dialogue_system/intent_router"
SLM_MODEL_ID           = "microsoft/Phi-3-mini-4k-instruct"

print("✅ Paths set")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ GPU: True
🖥️ Device: Tesla T4
✅ Paths set


In [3]:
from IPython.display import display, Javascript
from google.colab import output
import base64

def record_user_voice(filename="my_practice.wav"):
    js = Javascript("""
    async function recordAudio() {
      const div = document.createElement('div');
      const btn = document.createElement('button');
      const str = document.createElement('span');

      btn.textContent = '🎤 Click to Start Recording';
      btn.style.cssText = "padding:10px; background:#f44336; color:white; border:none; border-radius:5px; cursor:pointer; font-size:16px; margin:10px;";

      document.body.appendChild(div);
      div.appendChild(btn);
      div.appendChild(str);

      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      const recorder = new MediaRecorder(stream);
      let chunks = [];

      recorder.ondataavailable = (e) => chunks.push(e.data);
      recorder.onstop = async () => {
        const blob = new Blob(chunks, { type: 'audio/wav' });
        const reader = new FileReader();
        reader.readAsDataURL(blob);
        reader.onloadend = () => {
          window.audioData = reader.result.split(',')[1];
        };
      };

      btn.onclick = () => {
        if (recorder.state === 'inactive') {
          recorder.start();
          btn.textContent = '🛑 Stop Recording';
          btn.style.background = '#2196F3';
        } else {
          recorder.stop();
          btn.textContent = '✅ Processing...';
          btn.disabled = true;
        }
      };

      while (recorder.state !== 'inactive' || !window.audioData) {
        await new Promise(resolve => setTimeout(resolve, 100));
      }

      const data = window.audioData;
      window.audioData = null; // Clear for next time
      div.remove();
      return data;
    }

    // Attach to window so eval_js can find it
    window.recordAudio = recordAudio;
    """)

    display(js)
    print("Waiting for recording...")

    # Call the function we just defined on the window
    audio_data_base64 = output.eval_js('window.recordAudio()')

    with open(filename, "wb") as f:
        f.write(base64.b64decode(audio_data_base64))

    return filename

In [4]:
import os

# Path to your model folder
model_path = "/content/drive/MyDrive/manual_datasets/clovacall_data/final_asr_jamo_model"

# Potential naming issues:
# 1. Missing 'pre'
# 2. Missing '.json'
old_file = os.path.join(model_path, "processor_config.json") # or "processor_config.json"
new_file = os.path.join(model_path, "preprocessor_config.json")

if os.path.exists(old_file):
    os.rename(old_file, new_file)
    print(f"✅ Successfully renamed {old_file} to {new_file}")
else:
    print(f"❌ Could not find {old_file}. Check the folder again using !ls")

❌ Could not find /content/drive/MyDrive/manual_datasets/clovacall_data/final_asr_jamo_model/processor_config.json. Check the folder again using !ls


In [5]:
print("📦 Loading ASR model...")
asr_processor = Wav2Vec2Processor.from_pretrained(ASR_MODEL_PATH)
asr_model = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL_PATH).to("cuda")
asr_model.eval()
print("✅ ASR model loaded")

📦 Loading ASR model...
✅ ASR model loaded


In [6]:
print("📦 Loading Intent Router...")
router_tokenizer = AutoTokenizer.from_pretrained(ROUTER_PATH)
router_model = AutoModelForSequenceClassification.from_pretrained(ROUTER_PATH).to("cuda")
router_model.eval()
print("✅ Intent Router loaded")

📦 Loading Intent Router...
✅ Intent Router loaded


In [7]:
def compute_edit_distance(ref, hyp):
    N, M = len(ref), len(hyp)
    dp = [[0] * (M + 1) for _ in range(N + 1)]
    for i in range(N + 1): dp[i][0] = i
    for j in range(M + 1): dp[0][j] = j
    for i in range(1, N + 1):
        for j in range(1, M + 1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    return dp

def backtrack(dp, ref, hyp):
    i, j = len(ref), len(hyp)
    operations = []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i-1] == hyp[j-1]:
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            operations.append({"position": i, "type": "substitution",
                "expected": ref[i-1], "predicted": hyp[j-1]})
            i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            operations.append({"position": i, "type": "deletion",
                "expected": ref[i-1], "predicted": "[missing]"})
            i -= 1
        else:
            operations.append({"position": j, "type": "insertion",
                "expected": "[none]", "predicted": hyp[j-1]})
            j -= 1
    operations.reverse()
    return operations

def get_pronunciation_diagnostics(audio_input, target_hangul):
    """
    Takes audio (bytes or file path) and target sentence.
    Returns transcription, PER, and error breakdown.
    """
    # Load audio
    if isinstance(audio_input, str):
        speech, sr = torchaudio.load(audio_input)
    else:
        speech, sr = torchaudio.load(io.BytesIO(audio_input))

    # Resample if needed
    if sr != 16000:
        speech = torchaudio.transforms.Resample(sr, 16000)(speech)

    # Silence check
    if speech.abs().max().item() < 0.01:
        return "", 1.0, [], []

    # ASR inference
    input_values = asr_processor(
        speech.squeeze().numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    ).input_values.to("cuda")

    with torch.no_grad():
        logits = asr_model(input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_jamo_str = asr_processor.batch_decode(pred_ids)[0]
    transcription = jamotools.join_jamos(pred_jamo_str).strip()

    # PER calculation
    ref_clean  = target_hangul.replace(" ", "")
    hyp_clean  = transcription.replace(" ", "")
    ref_jamo   = list(jamotools.split_syllables(ref_clean))
    hyp_jamo   = list(jamotools.split_syllables(hyp_clean))

    dp          = compute_edit_distance(ref_jamo, hyp_jamo)
    total_edits = dp[len(ref_jamo)][len(hyp_jamo)]
    per         = total_edits / len(ref_jamo) if ref_jamo else 0.0

    ops         = backtrack(dp, ref_jamo, hyp_jamo)
    substitutions = sum(1 for o in ops if o["type"] == "substitution")
    deletions     = sum(1 for o in ops if o["type"] == "deletion")
    insertions    = sum(1 for o in ops if o["type"] == "insertion")

    syl_dp     = compute_edit_distance(list(ref_clean), list(hyp_clean))
    syl_errors = backtrack(syl_dp, list(ref_clean), list(hyp_clean))

    print(f"📝 Target:        {target_hangul}")
    print(f"📝 Transcription: {transcription}")
    print(f"📊 PER:           {per*100:.1f}%")
    print(f"📊 S={substitutions} D={deletions} I={insertions}")

    return transcription, per, syl_errors, (substitutions, deletions, insertions)

print("✅ ASR + PER functions defined")

✅ ASR + PER functions defined


In [8]:
def route_intent(text: str):
    encoding = router_tokenizer(
        text, max_length=64, padding='max_length',
        truncation=True, return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to("cuda")
    attention_mask = encoding['attention_mask'].to("cuda")

    with torch.no_grad():
        outputs = router_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=input_ids.new_zeros(1)
        )
    probs  = torch.softmax(outputs[1], dim=-1)
    pred   = torch.argmax(probs, dim=-1).item()
    score  = probs[0][pred].item()
    intent = "restaurant" if pred == 0 else "travel"
    return intent, score

def load_expert(expert_path: str):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        SLM_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="eager"
    )
    base_model.config.use_cache = False
    return PeftModel.from_pretrained(base_model, expert_path)

print("✅ Router + Expert functions defined")

✅ Router + Expert functions defined


In [30]:
def generate_tutor_response(transcription, per, syl_errors, error_counts, target_sentence):
    # 1. Routing & Loading
    intent, _ = route_intent(transcription)
    expert_model = load_expert(RESTAURANT_EXPERT_PATH if intent == "restaurant" else TRAVEL_EXPERT_PATH)
    slm_tokenizer = AutoTokenizer.from_pretrained(SLM_MODEL_ID, trust_remote_code=True)

    s, d, i = error_counts

    # 2. THE DETERMINISTIC PROMPT
    # Providing a "Correct" vs. "Mistake" example is the only way to ground a 3B model.
    prompt = f"""<|system|>
You are a Korean Language Coach. Speak ONLY in English.
Task: Provide 1 sentence of phonetic feedback based on the Metrics and 1 sentence of roleplay response.
Constraints: Do NOT use jargon. Do NOT suggest new Korean words.

Example:
Goal: "비빔밥 하나 주세요"
Student: "미빔밥 하나 주세요"
Metrics: S=1, D=0
<|assistant|>
Phonetic Feedback: You used an 'M' sound instead of a 'B' at the start; make sure to pop your lips for the 'B' sound. Coming right up! One bibimbap for you.

<|user|>
Goal: "{target_sentence}"
Student: "{transcription}"
Metrics: S={s}, D={d}, I={i}
<|assistant|>
Phonetic Feedback:"""

    inputs = slm_tokenizer(prompt, return_tensors="pt").to("cuda")

    # 3. Execution with Strict Constraints
    with torch.no_grad():
        outputs = expert_model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,      # Required for temperature
            temperature=0.1,    # Kills creativity/hallucinations
            repetition_penalty=1.2,
            tokenizer=slm_tokenizer,
            stop_strings=["\n", "<|end|>", "Goal:"]
        )

    # 4. Clean Extraction
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    response = slm_tokenizer.decode(generated, skip_special_tokens=True).strip()

    # Remove any internal labels
    clean_text = response.replace("Phonetic Feedback:", "").strip()
    full_output = f"Tutor Feedback: {clean_text}"

    # 5. GPU Cleanup
    del expert_model
    torch.cuda.empty_cache()

    return intent, full_output

In [31]:
# 🎯 1. Set your target sentence
target_sentence = "비빔밥 하나 주세요" # Or anything you want to practice!
print(f"🎯 Target: {target_sentence}")

# 🎤 2. Record yourself
# When you run this, a button will appear. Click it, speak, and click stop.
audio_input = record_user_voice("my_practice.wav")

# 📡 3. Step 1: Transcribe and score (Requires GPU)
print("\n📡 Step 1: Transcribing and scoring pronunciation...")
transcription, per, syl_errors, error_counts = get_pronunciation_diagnostics(
    audio_input, target_sentence
)

# 🤖 4. Step 2: Generate tutor response (Requires GPU)
print("\n🤖 Step 2: Generating tutor response...")
intent, response = generate_tutor_response(
    transcription, per, syl_errors, error_counts, target_sentence
)

print("\n" + "=" * 60)
print(f"🤖 Tutor Feedback: {response}")

🎯 Target: 비빔밥 하나 주세요


<IPython.core.display.Javascript object>

Waiting for recording...

📡 Step 1: Transcribing and scoring pronunciation...
📝 Target:        비빔밥 하나 주세요
📝 Transcription: 리빙바 바나주세요.
📊 PER:           22.2%
📊 S=2 D=1 I=1

🤖 Step 2: Generating tutor response...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


🤖 Tutor Feedback: Tutor Feedback: The pronunciation is slightly off with initial consonants - try again by emphasizing more lip roundness when saying '베이커리'. Here we go... A bowlful (of) delicious pork belly dish made from rice cake balls filled


In [ ]:
# 🎯 1. Set your target sentence
target_sentence = "화장실이 어디예요" # Or anything you want to practice!
print(f"🎯 Target: {target_sentence}")

# 🎤 2. Record yourself
# When you run this, a button will appear. Click it, speak, and click stop.
audio_input = record_user_voice("my_practice.wav")

# 📡 3. Step 1: Transcribe and score (Requires GPU)
print("\n📡 Step 1: Transcribing and scoring pronunciation...")
transcription, per, syl_errors, error_counts = get_pronunciation_diagnostics(
    audio_input, target_sentence
)

# 🤖 4. Step 2: Generate tutor response (Requires GPU)
print("\n🤖 Step 2: Generating tutor response...")
intent, response = generate_tutor_response(
    transcription, per, syl_errors, error_counts, target_sentence
)

print("\n" + "=" * 60)
print(f"🤖 Tutor Feedback: {response}")

🎯 Target: 화장실이 어디예요


<IPython.core.display.Javascript object>

Waiting for recording...


In [29]:
# 🎯 1. Set your target sentence
target_sentence = "떡볶이 좀 맵게 해주세요" # Or anything you want to practice!
print(f"🎯 Target: {target_sentence}")

# 🎤 2. Record yourself
# When you run this, a button will appear. Click it, speak, and click stop.
audio_input = record_user_voice("my_practice.wav")

# 📡 3. Step 1: Transcribe and score (Requires GPU)
print("\n📡 Step 1: Transcribing and scoring pronunciation...")
transcription, per, syl_errors, error_counts = get_pronunciation_diagnostics(
    audio_input, target_sentence
)

# 🤖 4. Step 2: Generate tutor response (Requires GPU)
print("\n🤖 Step 2: Generating tutor response...")
intent, response = generate_tutor_response(
    transcription, per, syl_errors, error_counts, target_sentence
)

print("\n" + "=" * 60)
print(f"🤖 Tutor Feedback: {response}")

🎯 Target: 떡볶이 좀 맵게 해주세요


<IPython.core.display.Javascript object>

Waiting for recording...

📡 Step 1: Transcribing and scoring pronunciation...
📝 Target:        떡볶이 좀 맵게 해주세요
📝 Transcription: 똑복기 종ㅁ ㅔㅂ개 해주세요.
📊 PER:           29.2%
📊 S=6 D=0 I=1

🤖 Step 2: Generating tutor response...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


🤖 Tutor Feedback: Tutor Feedback: Your pronunciation is excellent; thank you very much indeed
